## Step 1 — Confirm GPU + log environment

In [1]:
!nvidia-smi

Mon Dec 15 05:08:28 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import vllm
import sglang
print("vLLM OK, SGLang OK")

vLLM OK, SGLang OK


In [3]:
!fuser -k 8000/tcp || true

In [4]:
!nohup python -m vllm.entrypoints.openai.api_server \
  --model TinyLlama/TinyLlama-1.1B-Chat-v1.0 \
  --host 0.0.0.0 --port 8000 > vllm.log 2>&1 &

In [7]:
!tail -n 20 vllm.log

(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /v1/audio/transcriptions, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /v1/audio/translations, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /scale_elastic_ep, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /is_scaling_elastic_ep, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /inference/v1/generate, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /ping, Methods: GET
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /ping, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /invocations, Methods: POST
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /metrics, Methods: GET
(APIServer pid=21435) INFO 12-15 05:09:44 [launcher.py:46] Route: /classify, Methods: POST
(APIServer pid=21435) INFO 12-15 05:

In [8]:
!curl --max-time 5 -s -o /dev/null -w "%{http_code}\n" http://127.0.0.1:8000/health

200


In [9]:
import requests

url = "http://127.0.0.1:8000/v1/chat/completions"
payload = {
    "model": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "messages": [{"role": "user", "content": "Explain what a GPU does in one sentence."}],
    "max_tokens": 64,
    "temperature": 0.0
}

r = requests.post(url, json=payload, timeout=120)
print("Status:", r.status_code)
print(r.json()["choices"][0]["message"]["content"])

Status: 200
A GPU (Graphics Processing Unit) is a specialized processing unit designed to process graphics and video data. It is responsible for rendering images, animations, and other visual effects on a computer's display. The GPU is responsible for processing complex calculations and manipulating data, allowing for faster and more efficient graphics rendering


In [10]:
!tail -n 60 vllm.log

(EngineCore_DP0 pid=21563) 
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:22 [default_loader.py:308] Loading weights took 0.68 seconds
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:23 [gpu_model_runner.py:3549] Model loading took 2.0513 GiB memory and 1.859521 seconds
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:29 [backends.py:655] Using cache directory: /root/.cache/vllm/torch_compile_cache/9abc85caa8/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:29 [backends.py:715] Dynamo bytecode transform time: 5.77 s
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:35 [backends.py:216] Directly load the compiled graph(s) for dynamic shape from the cache, took 5.557 s
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:36 [monitor.py:34] torch.compile takes 11.33 s in total
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:37 [gpu_worker.py:359] Available KV cache memory: 68.99 GiB
(EngineCore_DP0 pid=21563) INFO 12-15 05:09:38 [kv_cache_utils.py:1286] GPU KV cache size: 3,288,20

## Step 2 - Run this benchmark script (vLLM)

### GPU logging in the background

In [11]:
!nohup bash -lc 'nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used,memory.total --format=csv -l 1 > gpu_util.csv' >/dev/null 2>&1 &
!echo "GPU logging started -> gpu_util.csv"

GPU logging started -> gpu_util.csv


### Benchmark script

In [12]:
import time, requests, pandas as pd

URL = "http://127.0.0.1:8000/v1/chat/completions"
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

PROMPTS = [
    "Explain cloud computing in one sentence.",
    "Summarize: model serving platforms need to balance latency, throughput, and cost.",
    "Write 3 bullet points comparing CPU vs GPU for inference.",
    "Explain what batching means in inference and why it improves throughput.",
]

def run_once(prompt, max_tokens=128, temperature=0.0):
    payload = {
        "model": MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": temperature,
    }
    t0 = time.time()
    r = requests.post(URL, json=payload, timeout=180)
    t1 = time.time()
    r.raise_for_status()
    data = r.json()
    text = data["choices"][0]["message"]["content"]
    # vLLM/OpenAI response often includes usage; use it if available
    usage = data.get("usage", {})
    return {
        "latency_s": t1 - t0,
        "output_text": text,
        "prompt_tokens": usage.get("prompt_tokens", None),
        "completion_tokens": usage.get("completion_tokens", None),
        "total_tokens": usage.get("total_tokens", None),
    }

rows = []

# Warm-up (important)
_ = run_once("Say hello.", max_tokens=32)

for max_toks in [64, 128, 256]:
    for prompt in PROMPTS:
        for trial in range(5):
            out = run_once(prompt, max_tokens=max_toks)
            completion_tokens = out["completion_tokens"]
            # If token count isn't provided, approximate by word count (fallback)
            approx_tokens = len(out["output_text"].split())
            gen_tokens = completion_tokens if completion_tokens is not None else approx_tokens
            rows.append({
                "framework": "vLLM",
                "model": MODEL,
                "max_tokens": max_toks,
                "prompt_len_chars": len(prompt),
                "trial": trial,
                "latency_s": out["latency_s"],
                "gen_tokens": gen_tokens,
                "tokens_per_sec": gen_tokens / out["latency_s"] if out["latency_s"] > 0 else None,
            })

df_vllm = pd.DataFrame(rows)
df_vllm.head()

,framework,model,max_tokens,prompt_len_chars,trial,latency_s,gen_tokens,tokens_per_sec
0,vLLM,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,0,0.192914,52,269.550150
1,vLLM,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,1,0.180497,52,288.093160
2,vLLM,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,2,0.179786,52,289.233191
3,vLLM,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,3,0.180546,52,288.015170
4,vLLM,TinyLlama/TinyLlama-1.1B-Chat-v1.0,64,40,4,0.181047,52,287.217917


In [13]:
df_vllm.to_csv("vllm_tinyllama_results.csv", index=False)
print("Saved:", "vllm_tinyllama_results.csv")
df_vllm.groupby("max_tokens")[["latency_s","tokens_per_sec"]].mean()

Saved: vllm_tinyllama_results.csv


,latency_s,tokens_per_sec
max_tokens,,
64,0.211416,288.398043
128,0.335330,292.125836
256,0.537353,287.163449


### Stop GPU logging

In [14]:
!pkill -f "nvidia-smi --query-gpu" || true
!echo "GPU logging stopped"

^C
GPU logging stopped
